# 03 Modeling

This notebook trains and evaluates a Ridge Regression model for job-level mean double-shear strength. A mean baseline model is included for comparison, and validation is grouped by material lot to avoid splitting related lot-level records across train and test sets.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Make paths work whether the notebook is run from the repo root or notebooks folder
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    REPO_ROOT = NOTEBOOK_DIR

DATA_DIR = REPO_ROOT / "data"
FIGURE_DIR = REPO_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)

processed_path = DATA_DIR / "processed_material_database.csv"

from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor


## Load processed dataset and feature list

In [ ]:
df = pd.read_csv(processed_path)
feature_info = pd.read_csv(DATA_DIR / "feature_list.csv")

numeric_features = feature_info.loc[feature_info["type"] == "numeric", "feature"].tolist()
categorical_features = feature_info.loc[feature_info["type"] == "categorical", "feature"].tolist()

target_col = "MeanShear_ksi"
group_col = "Lot_ID"

print("Dataset shape:")
print(df.shape)

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


In [ ]:
model_cols = numeric_features + categorical_features + [target_col, group_col]
model_df = df[model_cols].copy()

# Fill missing values in a simple, transparent way
for col in numeric_features:
    model_df[col] = model_df[col].fillna(model_df[col].median())

for col in categorical_features:
    model_df[col] = model_df[col].fillna("Unknown")

model_df = model_df.dropna(subset=[target_col, group_col]).reset_index(drop=True)

X = model_df[numeric_features + categorical_features]
y = model_df[target_col]
groups = model_df[group_col]

print("Model dataframe shape:")
print(model_df.shape)

print("\nUnique lots:")
print(groups.nunique())

display(model_df.head())


## Define baseline and Ridge Regression models

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

baseline_model = DummyRegressor(strategy="mean")

ridge_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", Ridge(alpha=1.0))
    ]
)


## Group-aware cross-validation

The validation split uses `Lot_ID` as the group key. 

In [ ]:
unique_lots = groups.nunique()

if unique_lots < 2:
    raise ValueError("At least two unique lots are required for LeaveOneGroupOut validation.")

cv = LeaveOneGroupOut()

y_pred_baseline = cross_val_predict(
    baseline_model,
    X,
    y,
    cv=cv,
    groups=groups
)

y_pred_ridge = cross_val_predict(
    ridge_model,
    X,
    y,
    cv=cv,
    groups=groups
)

baseline_mae = mean_absolute_error(y, y_pred_baseline)
baseline_rmse = np.sqrt(mean_squared_error(y, y_pred_baseline))
baseline_r2 = r2_score(y, y_pred_baseline)

ridge_mae = mean_absolute_error(y, y_pred_ridge)
ridge_rmse = np.sqrt(mean_squared_error(y, y_pred_ridge))
ridge_r2 = r2_score(y, y_pred_ridge)

results = pd.DataFrame({
    "Model": ["Mean baseline", "Ridge Regression"],
    "MAE_ksi": [baseline_mae, ridge_mae],
    "RMSE_ksi": [baseline_rmse, ridge_rmse],
    "R2": [baseline_r2, ridge_r2]
})

display(results)
results.to_csv(DATA_DIR / "model_results.csv", index=False)
print("Saved model results to data/model_results.csv")


## Save prediction table

In [ ]:
prediction_table = model_df[[group_col, target_col]].copy()

if "TestStage" in model_df.columns:
    prediction_table["TestStage"] = model_df["TestStage"]

if "AgeTemp_F" in model_df.columns:
    prediction_table["AgeTemp_F"] = model_df["AgeTemp_F"]

prediction_table["Pred_Baseline_ksi"] = y_pred_baseline
prediction_table["Pred_Ridge_ksi"] = y_pred_ridge
prediction_table["Ridge_Error_ksi"] = prediction_table["Pred_Ridge_ksi"] - prediction_table[target_col]
prediction_table["Ridge_AbsError_ksi"] = prediction_table["Ridge_Error_ksi"].abs()

display(prediction_table)

prediction_table.to_csv(DATA_DIR / "model_predictions.csv", index=False)
print("Saved predictions to data/model_predictions.csv")


## Prediction comparison plot

In [ ]:
plt.figure(figsize=(6, 6))

plt.scatter(y, y_pred_baseline, label="Mean baseline", s=70)
plt.scatter(y, y_pred_ridge, label="Ridge Regression", s=70)

min_val = min(y.min(), y_pred_baseline.min(), y_pred_ridge.min())
max_val = max(y.max(), y_pred_baseline.max(), y_pred_ridge.max())

plt.plot([min_val, max_val], [min_val, max_val], "k--", label="Ideal prediction")

plt.xlabel("Actual mean shear strength (ksi)")
plt.ylabel("Predicted mean shear strength (ksi)")
plt.title("Proof-of-concept prediction comparison")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = FIGURE_DIR / "figure2_prediction_comparison.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {fig_path}")


## Fit final model for screening demonstration

The final model is fit on all available rows.

In [ ]:
ridge_model.fit(X, y)
print("Final proof-of-concept Ridge model fit on all available data.")


In [ ]:
# Ridge coefficient feature importance plot

ridge_model.fit(X, y)

# Get feature names after preprocessing
feature_names = ridge_model.named_steps["preprocess"].get_feature_names_out()

# Get Ridge coefficients
ridge_coefficients = ridge_model.named_steps["model"].coef_

# Build coefficient table
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": ridge_coefficients
})

coef_df["AbsCoefficient"] = coef_df["Coefficient"].abs()

# Clean feature names for plotting
coef_df["Feature"] = (
    coef_df["Feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

# Select top features
top_coef = coef_df.sort_values("AbsCoefficient", ascending=False).head(10)

# Plot
plt.figure(figsize=(7, 4))
plt.barh(top_coef["Feature"], top_coef["AbsCoefficient"])
plt.xlabel("Absolute standardized Ridge coefficient")
plt.ylabel("Feature")
plt.title("Top Ridge Regression coefficient magnitudes")
plt.gca().invert_yaxis()
plt.tight_layout()

figure_path = FIGURE_DIR / "figure3_ridge_feature_importance.png"
plt.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {figure_path}")